# Basic Text Reflection — Generate and Revise a LinkedIn Post

This notebook follows the essay workflow pattern to generate, critique,
and revise a LinkedIn post.


In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Read values
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

# Create client
client = OpenAI(
    api_key=api_key,
    base_url=base_url
)


DEFAULT_MODEL = "gpt-4o-mini"  

## Part 1 — The Three Core Functions

In [3]:
def generate_draft(topic, tone="professional", model=DEFAULT_MODEL):
    """Generates a first draft of a LinkedIn post."""
    prompt = f"""Write a LinkedIn post about "{topic}".

Tone: {tone}

Mandatory constraints:
- Maximum 300 characters (entire post)
- Exactly 3 to 5 relevant hashtags at the end of the post

Return only the post text, with no extra explanation and no quotation marks."""

    response = client.chat.completions.create(
        model=model,
        temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


In [4]:
def reflect_on_draft(draft, model=DEFAULT_MODEL):
    """Gives general feedback and critique on a LinkedIn post draft."""
    prompt = f"""Below is a draft of a LinkedIn post:

---
{draft}
---

Critique this post on the following points:
- Is the length appropriate for a good LinkedIn post?
- Is the number and quality of hashtags appropriate?
- Is the tone engaging and suitable for a professional audience?

Keep the feedback short, a few sentences."""

    response = client.chat.completions.create(
        model=model,
        temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


In [5]:
def revise_draft(draft, feedback, model=DEFAULT_MODEL):
    """Revises the post based on the feedback received."""
    prompt = f"""Original LinkedIn post draft:
---
{draft}
---

Feedback received on this draft:
---
{feedback}
---

Based on this feedback, write a revised version of the post.
The constraints of max 300 characters and 3 to 5 hashtags must be respected.
Return only the final post text."""

    response = client.chat.completions.create(
        model=model,
        temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


## Combined Function to Run the Full Workflow

In [6]:
def run_linkedin_workflow(topic, tone="professional", model=DEFAULT_MODEL):
    """Runs all three steps (generate, reflect, revise) in sequence."""
    draft = generate_draft(topic, tone=tone, model=model)
    feedback = reflect_on_draft(draft, model=model)
    revised = revise_draft(draft, feedback, model=model)

    return {
        "topic": topic,
        "draft": draft,
        "feedback": feedback,
        "revised": revised,
    }


## Test with Two Different Topics

In [7]:
result_1 = run_linkedin_workflow("announcing a new AI training course")
result_2 = run_linkedin_workflow("announcing a work achievement: successful completion of a major project")

for result in [result_1, result_2]:
    print("=" * 60)
    print("Topic:", result["topic"])
    print("-" * 60)
    print("First draft:\n", result["draft"])
    print("-" * 60)
    print("Feedback:\n", result["feedback"])
    print("-" * 60)
    print("Final version:\n", result["revised"])
    print()


Topic: announcing a new AI training course
------------------------------------------------------------
First draft:
 Excited to announce the launch of our new AI training course! This comprehensive program is designed to equip professionals with the essential skills needed to excel in AI applications. Join us in harnessing the power of artificial intelligence and elevating your career to new heights! #AITraining #ProfessionalDevelopment #ArtificialIntelligence #CareerGrowth #SkillDevelopment
------------------------------------------------------------
Feedback:
 The length of the post is appropriate for LinkedIn, as it provides a clear message without being overly lengthy. However, consider adding a call-to-action, such as a link to the course or an invitation to sign up, to encourage engagement. The number of hashtags is reasonable, but you might want to focus on more specific ones relevant to your audience, as this can enhance visibility. The tone is engaging and professional, succe

### Discussion Question for Your Report

Why is `temperature=1.0` used for the generation and reflection steps in the
essay workflow, while `temperature=0` is used for SQL generation (02_02)?
How does this difference relate to the creative vs. deterministic nature of
each task?

**Your answer:**

_(write here)_


---
## Part 2 — Adding External Feedback

In [8]:
def check_post_quality(post_text, max_length=300, min_hashtags=3, max_hashtags=5):
    """Checks post quality using simple Python rules (no LLM call)."""
    length = len(post_text)
    within_length_limit = length <= max_length

    hashtags = [word for word in post_text.split() if word.startswith("#")]
    hashtag_count = len(hashtags)
    within_hashtag_range = min_hashtags <= hashtag_count <= max_hashtags

    return {
        "length": length,
        "within_length_limit": within_length_limit,
        "hashtag_count": hashtag_count,
        "within_hashtag_range": within_hashtag_range,
    }


In [9]:
# Quick test of the function on a manual sample
sample_check = check_post_quality("This is a sample post without enough hashtags.")
sample_check


{'length': 46,
 'within_length_limit': True,
 'hashtag_count': 0,
 'within_hashtag_range': False}

## New Reflect Version That Sees the Quality Signals

In [10]:
def reflect_on_draft_with_signals(draft, quality_signals, model=DEFAULT_MODEL):
    """Like reflect_on_draft, but includes the quality_signals dictionary in the prompt."""
    prompt = f"""Below is a draft of a LinkedIn post:

---
{draft}
---

Automated quality check results for this post:
- Post length: {quality_signals['length']} characters (within limit? {quality_signals['within_length_limit']})
- Hashtag count: {quality_signals['hashtag_count']} (within range? {quality_signals['within_hashtag_range']})

Using these exact numbers, point out the post's issues explicitly.
If any constraint is violated, state exactly by how much and why it's a problem."""

    response = client.chat.completions.create(
        model=model,
        temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


## Comparison: Feedback With Signals vs. Without Signals

In [11]:
# An intentionally flawed post: no hashtags at all and slightly too long
bad_post = (
    "Today I want to talk about a major achievement. "
    "After months of team effort, our big project was finally completed successfully "
    "and delivered fantastic results for the business. "
    "Thank you to everyone who was part of this journey."
)

quality_signals = check_post_quality(bad_post)
print("Quality signals:", quality_signals)

feedback_without_signals = reflect_on_draft(bad_post)
feedback_with_signals = reflect_on_draft_with_signals(bad_post, quality_signals)

print("=" * 60)
print("Feedback without quality signals:\n", feedback_without_signals)
print("=" * 60)
print("Feedback with quality signals:\n", feedback_with_signals)


Quality signals: {'length': 229, 'within_length_limit': True, 'hashtag_count': 0, 'within_hashtag_range': False}
Feedback without quality signals:
 The length of the post is suitable for LinkedIn, as it's concise and to the point, which is ideal for professional networking. However, it would benefit from more specific details about the project and its impact, making it more engaging. The post currently lacks hashtags, which can help increase visibility; consider adding a few relevant ones related to your industry or project. Overall, the tone is positive and professional, but it could be made more dynamic by sharing specific achievements or challenges overcome.
Feedback with quality signals:
 The automated quality check results indicate two issues with the LinkedIn post:

1. **Post Length**: The post is 229 characters long, which is within the typical LinkedIn character limit (which allows for up to 3,000 characters). This part of the evaluation is fine, so there is no issue here.

2. 

### Conclusion

The version that sees `quality_signals` usually points directly to exact
numbers (e.g. "you have 0 hashtags, at least 3 are required"), while the
version without signals usually only makes a vague remark like "not enough
hashtags" or "the post is a bit long".
